# Neo4j MCP Agent with Strands Agents

This notebook demonstrates how to build an AI agent that queries a Neo4j knowledge graph using the **Model Context Protocol (MCP)** with the **Strands Agents SDK**.

Strands provides a lightweight, AWS-native approach with built-in MCP support. Compared to the LangGraph notebook, this version requires fewer lines of code and has a simpler API.

**What you'll do:**
- Connect to a Neo4j MCP Server through an AWS AgentCore Gateway
- Discover available tools using the Strands MCPClient
- Create an agent that queries SEC financial data with natural language

**Prerequisites:**
- Lab 1 complete (Neo4j Aura instance with SEC data loaded)
- `CONFIG.txt` updated with `MCP_GATEWAY_URL` and `MCP_ACCESS_TOKEN`

## 1. Setup

Install the required packages for the Strands MCP agent.

In [ ]:
%pip install strands-agents strands-agents-tools mcp httpx -q

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

print("All imports successful.")

## 2. Configuration

Load settings from `CONFIG.txt`. The MCP Gateway URL and access token authenticate your connection to the Neo4j MCP Server running on AWS.

In [ ]:
# Load configuration from CONFIG.txt
config = {}
with open("../CONFIG.txt", "r") as f:
    for line in f:
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            config[key.strip()] = value.strip()

MODEL_ID = config.get("MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
REGION = config.get("REGION", "us-east-1")
GATEWAY_URL = config.get("MCP_GATEWAY_URL", "")
ACCESS_TOKEN = config.get("MCP_ACCESS_TOKEN", "")

print(f"Model ID:       {MODEL_ID}")
print(f"Region:         {REGION}")
print(f"Gateway URL:    {GATEWAY_URL[:50]}..." if len(GATEWAY_URL) > 50 else f"Gateway URL:    {GATEWAY_URL}")
print(f"Access Token:   {'[SET]' if ACCESS_TOKEN else '[NOT SET]'}")

# Validate required settings
assert GATEWAY_URL and GATEWAY_URL != "your-gateway-url-here", \
    "MCP_GATEWAY_URL not configured in CONFIG.txt. Update it with your AgentCore Gateway URL."
assert ACCESS_TOKEN and ACCESS_TOKEN != "your-access-token-here", \
    "MCP_ACCESS_TOKEN not configured in CONFIG.txt. Update it with your access token."

## 3. Initialize Model & MCP Client

Create the Bedrock model and MCP client:

- **BedrockModel**: Configures Claude on Amazon Bedrock with temperature 0 for deterministic responses
- **MCPClient**: Manages the connection to the Neo4j MCP Server. It takes a transport factory function that creates a new Streamable HTTP connection with authentication headers each time the client is started.

In [ ]:
# Initialize the Bedrock model
bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=REGION,
    temperature=0,
)

print(f"Model initialized: {MODEL_ID}")


# Define authentication token provider
def get_token() -> str:
    """Return the MCP access token for gateway authentication."""
    return ACCESS_TOKEN


# Define the transport factory for the MCP client
def create_streamable_http_transport():
    """Create a Streamable HTTP transport with authentication headers."""
    return streamablehttp_client(
        url=GATEWAY_URL,
        headers={"Authorization": f"Bearer {get_token()}"},
    )


# Create the MCP client
mcp_client = MCPClient(transport_callable=create_streamable_http_transport)

print("MCP client created.")

## 4. Test MCP Connection

Verify the MCP connection by listing the available tools. The Neo4j MCP Server should expose tools for schema retrieval and Cypher query execution.

In [ ]:
with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Connected! Found {len(tools)} tools:\n")
    for tool in tools:
        print(f"  - {tool.tool_name}: {tool.tool_description}")

## 5. Create Agent & Query Function

Define the system prompt and a `query()` function that:

1. Opens an MCP connection to discover tools
2. Creates a Strands Agent with the Bedrock model, system prompt, and MCP tools
3. Sends the user's question to the agent and prints the response

In [ ]:
SYSTEM_PROMPT = """You are a Neo4j database assistant with access to a knowledge graph 
containing SEC 10-K financial filing data (companies, products, services, risk factors, 
financial metrics, executives, asset manager holdings).

Rules:
1. Always retrieve the database schema before writing Cypher queries.
2. Only use read-only Cypher (MATCH, RETURN, WITH, WHERE, ORDER BY, LIMIT).
3. Include LIMIT clauses to avoid excessive results.
4. Use COALESCE() or IS NOT NULL for properties that might be missing.
5. Format results clearly and cite actual data from query results.
"""


def query(question: str) -> None:
    """Ask a natural language question about the Neo4j database."""
    print(f"Question: {question}\n")

    with mcp_client:
        tools = mcp_client.list_tools_sync()
        print(f"  Tools: {[t.tool_name for t in tools]}")

        agent = Agent(
            model=bedrock_model,
            system_prompt=SYSTEM_PROMPT,
            tools=tools,
        )

        response = agent(question)
        print(f"\nAnswer:\n{response}")


print("Query function ready.")

## 6. Demo Queries

Run the following queries to see the agent discover the schema and query the graph. Watch the tool calls in the output to understand how the agent reasons about each question.

In [ ]:
query("What is the database schema?")

In [ ]:
query("How many nodes are there by label?")

In [ ]:
query("Show 5 sample records from the most populated node type.")

## 7. Your Query

Try your own questions about the SEC financial data:

- "How many companies are in the database?"
- "What products does Apple offer?"
- "Which asset managers own stakes in NVIDIA?"
- "What risk factors does Microsoft face?"
- "Show me the financial metrics for Tesla."
- "Who are the executives at Amazon?"
- "Which companies face risk factors related to cybersecurity?"

In [ ]:
# query("Your question here")